In [2]:
from google.colab import files
uploaded = files.upload()

Saving 7832c05caae3489cbcbbb9b02cf61711.parquet to 7832c05caae3489cbcbbb9b02cf61711.parquet


In [3]:
import pandas as pd

# Replace with your actual filename
file_name = list(uploaded.keys())[0]

df = pd.read_parquet(file_name)

df.head()

,time_id,seconds_in_bucket,bid_price1,ask_price1,bid_price2,ask_price2,bid_size1,ask_size1,bid_size2,ask_size2
0,4,0,1.000049,1.000590,0.999656,1.000639,91,100,100,24
1,4,1,1.000049,1.000590,0.999656,1.000639,91,100,100,20
2,4,5,1.000049,1.000639,0.999656,1.000885,290,20,101,15


In [4]:
# Column names
print(df.columns)

# Data info
df.info()

# Summary stats
df.describe()

Index(['time_id', 'seconds_in_bucket', 'bid_price1', 'ask_price1',
       'bid_price2', 'ask_price2', 'bid_size1', 'ask_size1', 'bid_size2',
       'ask_size2'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   time_id            3 non-null      int16  
 1   seconds_in_bucket  3 non-null      int16  
 2   bid_price1         3 non-null      float32
 3   ask_price1         3 non-null      float32
 4   bid_price2         3 non-null      float32
 5   ask_price2         3 non-null      float32
 6   bid_size1          3 non-null      int32  
 7   ask_size1          3 non-null      int32  
 8   bid_size2          3 non-null      int32  
 9   ask_size2          3 non-null      int32  
dtypes: float32(4), int16(2), int32(4)
memory usage: 240.0 bytes


,time_id,seconds_in_bucket,bid_price1,ask_price1,bid_price2,ask_price2,bid_size1,ask_size1,bid_size2,ask_size2
count,3.0,3.000000,3.000000,3.000000,3.000000e+00,3.000000,3.000000,3.000000,3.000000,3.000000
mean,4.0,2.000000,1.000049,1.000606,9.996559e-01,1.000721,157.333333,73.333333,100.333333,19.666667
std,0.0,2.645751,0.000000,0.000028,7.300049e-08,0.000142,114.892704,46.188022,0.577350,4.509250
min,4.0,0.000000,1.000049,1.000590,9.996558e-01,1.000639,91.000000,20.000000,100.000000,15.000000
25%,4.0,0.500000,1.000049,1.000590,9.996558e-01,1.000639,91.000000,60.000000,100.000000,17.500000
50%,4.0,1.000000,1.000049,1.000590,9.996558e-01,1.000639,91.000000,100.000000,100.000000,20.000000
75%,4.0,3.000000,1.000049,1.000615,9.996558e-01,1.000762,190.500000,100.000000,100.500000,22.000000
max,4.0,5.000000,1.000049,1.000639,9.996558e-01,1.000885,290.000000,100.000000,101.000000,24.000000


# Book Testing and Training Data

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
base_path = '/content/drive/MyDrive/Optiver project 4 data'

In [7]:
!pip install pyarrow

In [10]:
df.head()
df.shape
df['time_id'].nunique()

1

In [11]:
# Spread: profit opportunity for market makers
df['spread'] = df['ask_price1'] - df['bid_price1']

# Weighted Average Price (WAP): better estimate of fair price
df['wap'] = (
    df['bid_price1'] * df['ask_size1'] +
    df['ask_price1'] * df['bid_size1']
) / (df['bid_size1'] + df['ask_size1'])

# Order imbalance: demand vs supply pressure
df['imbalance'] = (
    df['bid_size1'] - df['ask_size1']
) / (df['bid_size1'] + df['ask_size1'])

What this does:

- Converts raw order book → tradable signals
- These are the exact features used in HFT models

In [12]:
import numpy as np

df['log_return'] = df.groupby(['time_id'])['wap'].transform(
    lambda x: np.log(x).diff()
)

In [13]:
realized_vol = df.groupby('time_id')['log_return'].apply(
    lambda x: np.sqrt(np.sum(x**2))
).reset_index()

realized_vol.columns = ['time_id', 'realized_vol']

What this does:
1. Measures price movement
2. Foundation for volatility
3. Removes scale effects


What Realized volatility does:
1. Converts high-frequency moves → volatility
2. This is the target Optiver wants to predict.

This formula is actually a discrte approximation of the integral of the volatility squared and measured over time period t, which makes a foundation of Black- Scholes, Stochastic Calculus and Realised Volatility Theory.

In [14]:
features = df.groupby('time_id').agg({
    'spread': 'mean',
    'wap': 'mean',
    'imbalance': 'mean',
    'bid_size1': 'mean',
    'ask_size1': 'mean'
}).reset_index()

In [15]:
final_df = features.merge(realized_vol, on='time_id')

What this does:
- Creates supervised learning dataset
- X = features
- y = realized volatility

In [16]:
final_df.head()
final_df.describe()

,time_id,spread,wap,imbalance,bid_size1,ask_size1,realized_vol
count,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
mean,4.0,0.000557,1.000405,0.258909,157.333333,73.333333,0.000294
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,4.0,0.000557,1.000405,0.258909,157.333333,73.333333,0.000294
25%,4.0,0.000557,1.000405,0.258909,157.333333,73.333333,0.000294
50%,4.0,0.000557,1.000405,0.258909,157.333333,73.333333,0.000294
75%,4.0,0.000557,1.000405,0.258909,157.333333,73.333333,0.000294
max,4.0,0.000557,1.000405,0.258909,157.333333,73.333333,0.000294
